# Analytics Snippets

Small patterns you can copy into the workbook if you get stuck. Replace the column names with the ones used in the task cell. If you already know the pattern, work straight in the participant notebook.

In [ ]:
import pandas as pd

df = pd.read_csv('messy_turbine_telemetry.csv', parse_dates=['Timestamp'])
print(df.head())
print(df.info())
print(df.isna().sum())

In [ ]:
import numpy as np
import pandas as pd

# Keep dropout rows as forensic evidence; build a separate clean stream for charts.
df['Failure_State'] = np.where(
    df[['Bearing_Temperature_C', 'Vibration_mm_s']].isna().any(axis=1).values,
    'Sensor dropout / failure',
    'Normal',
)
failure_records = df[df['Failure_State'] == 'Sensor dropout / failure'].copy()
analysis_df = df.dropna(subset=['Bearing_Temperature_C', 'Vibration_mm_s']).copy()
analysis_df = analysis_df.sort_values(['Turbine_ID', 'Timestamp'])

# IQR filter (3×) to remove injected extreme spikes before plotting.
for col in ['Bearing_Temperature_C', 'Vibration_mm_s']:
    Q1, Q3 = analysis_df[col].quantile(0.25), analysis_df[col].quantile(0.75)
    IQR = Q3 - Q1
    analysis_df = analysis_df[
        (analysis_df[col] >= Q1 - 3 * IQR) & (analysis_df[col] <= Q3 + 3 * IQR)
    ]

print('Dropout rows kept for forensic review:', len(failure_records))
print('Rows remaining for analysis:', len(analysis_df))

# Vibration normalised by RPM — filter to running turbine before dividing.
analysis_df['Vibration_per_RPM'] = (
    analysis_df['Vibration_mm_s'] / analysis_df['RPM'].replace(0, pd.NA)
)
analysis_df['Vibration_per_RPM_Rolling'] = (
    analysis_df.groupby('Turbine_ID')['Vibration_per_RPM']
    .transform(lambda s: s.rolling(window=24, min_periods=6).mean())
)

In [ ]:
import plotly.express as px

fig = px.scatter(
    clean_df,
    x='Bearing_Temperature_C',
    y='Vibration_mm_s',
    color='Turbine_ID',
    size='RPM',
    hover_data=['Wind_Speed_mps', 'Wind_Gust_mps', 'Turbulence_Intensity'],
)
fig.show()

fig = px.line(clean_df, x='Timestamp', y='Bearing_Temperature_C', color='Turbine_ID')
fig.update_traces(connectgaps=False)
fig.show()

In [ ]:
summary = clean_df.groupby('Turbine_ID').agg({
    'Power_kW': 'mean',
    'Vibration_mm_s': 'max',
    'Maintenance_Flag': 'last',
})
print(summary)

# Use the summary table with one chart to explain the maintenance choice.
# Keep the story tied to the sensor readings, not just the chart shape.